In [5]:
import yfinance as yf
import numpy as np
from scipy.optimize import minimize

# 1. Загрузка данных
tickers = ["AAPL", "TSLA"]
# yfinance >= 1.0: auto_adjust=True by default, adjusted prices are in "Close"
data = yf.download(tickers, start="2015-01-01", end="2025-01-01", interval="1mo")["Close"]
test_data = yf.download(tickers, start="2025-01-01", end="2025-12-31", interval="1mo")["Close"]

# 2. Оценка параметров (Train)
returns = np.log(data / data.shift(1)).dropna()
mu = returns.mean().values
Sigma = returns.cov().values
rf_monthly = 0.035 / 12

# 3. Оптимизация (Kelly Criterion)
def negative_kelly(w, mu, Sigma, rf):
    port_return = rf + np.dot(w, mu - rf)
    port_var = np.dot(w.T, np.dot(Sigma, w))
    return -(port_return - 0.5 * port_var)

init_guess = np.array([0.5, 0.5])
res = minimize(negative_kelly, init_guess, args=(mu, Sigma, rf_monthly), method='BFGS')
w_kelly = res.x

print(f"Оптимальные веса по Келли (AAPL, TSLA): {w_kelly.round(4)}")
print(f"Доля в безрисковом активе: {(1 - np.sum(w_kelly)):.4f}")

# 4. Бэктест (Test)
test_returns = test_data.pct_change().dropna()
port_test_returns = test_returns.dot(w_kelly) + (1 - np.sum(w_kelly)) * rf_monthly

norm_return = np.prod(1 + port_test_returns) - 1
test_mean = port_test_returns.mean()
test_std = port_test_returns.std()
sharpe_ratio = (test_mean - rf_monthly) / test_std * np.sqrt(12) # Аннуализированный

print(f"Норма доходности за тестовый период: {norm_return:.4f}")
print(f"Sharpe Ratio (annualized): {sharpe_ratio:.4f}")

[*********************100%***********************]  1 of 2 completed
[*********************100%***********************]  2 of 2 completed

Оптимальные веса по Келли (AAPL, TSLA): [2.1732 0.3858]
Доля в безрисковом активе: -1.5589
Норма доходности за тестовый период: 0.3217
Sharpe Ratio (annualized): 0.7250
